# Exercise 2 — Waitlist

A waitlist is the simplest way to validate demand before building. The `Waitlist` class stores subscribers in a JSON file on disk. It deduplicates by email, normalises to lowercase, validates the `@` character, and can export to CSV for use in email tools like Mailchimp or ConvertKit.

In [ ]:
import csv, datetime, io, json, pathlib, tempfile
from dataclasses import dataclass, field

@dataclass
class LandingPageConfig:
    product_name: str; tagline: str; description: str; features: list
    cta_text: str = "Join the Waitlist"; cta_url: str = "#waitlist"
    primary_color: str = "#2563eb"

_CFG = LandingPageConfig(
    product_name  = "AI Trading Bot",
    tagline       = "Automate your trading strategy with AI-powered signals.",
    description   = "Built with sentiment analysis, technical indicators, and risk controls.",
    features      = ["Sentiment-driven signals", "Stop-loss protection", "Daily scheduling"],
)

@dataclass
class WaitlistEntry:
    """One waitlist subscriber."""
    email:     str
    name:      str = ""
    joined_at: str = field(default_factory=lambda: datetime.datetime.now().isoformat())
    source:    str = "landing_page"


class Waitlist:
    """File-backed waitlist manager.

    Stores subscribers as a JSON array at self.path.
    add()        — add email; return True if added, False if duplicate
    count()      — number of subscribers
    list_emails() — list of email strings
    export_csv() — CSV string with columns: email, name, joined_at, source

    Raises ValueError for invalid emails (empty or no '@').
    Strips and lowercases emails before storing.
    """
    def __init__(self, path):
        self.path = pathlib.Path(path)
        self._entries = []
        if self.path.exists():
            self._load()

    def _load(self):
        # TODO: load JSON from self.path; populate self._entries
        pass

    def _save(self):
        # TODO: mkdir parents; write JSON to self.path
        pass

    def add(self, email, name="", source="landing_page"):
        # TODO: strip+lower email; validate; deduplicate; append; save; return bool
        return False

    def count(self):
        # TODO: 1 line
        return 0

    def list_emails(self):
        # TODO: 1 line
        return []

    def export_csv(self):
        # TODO: use csv.DictWriter with io.StringIO; return string
        return ""


### Checks

In [ ]:
checks = 0

with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
    wl_path = f.name

try:
    import os; os.unlink(wl_path)  # start with no file
except FileNotFoundError:
    pass

# 1 — add returns True for new email, count increments
try:
    wl = Waitlist(wl_path)
    r1 = wl.add("alice@example.com", name="Alice")
    r2 = wl.add("bob@example.com",   name="Bob")
    assert r1 is True and r2 is True, f"expected True, True; got {r1}, {r2}"
    assert wl.count() == 2, f"expected count=2, got {wl.count()}"
    checks += 1; print("✅ 1 add() returns True, count() = 2")
except Exception as e:
    print("❌ 1:", e)

# 2 — duplicate returns False, count unchanged
try:
    wl = Waitlist(wl_path)
    r3 = wl.add("alice@example.com")
    assert r3 is False, f"expected False for duplicate, got {r3}"
    assert wl.count() == 2
    checks += 1; print("✅ 2 duplicate add() returns False, count stays 2")
except Exception as e:
    print("❌ 2:", e)

# 3 — email normalisation (uppercase → lowercase)
try:
    wl = Waitlist(wl_path)
    r4 = wl.add("CAROL@Example.COM", name="Carol")
    assert r4 is True
    assert "carol@example.com" in wl.list_emails()
    checks += 1; print("✅ 3 email normalised to lowercase: carol@example.com")
except Exception as e:
    print("❌ 3:", e)

# 4 — persistence: reload from file
try:
    wl2 = Waitlist(wl_path)   # fresh instance, same file
    emails = wl2.list_emails()
    assert "alice@example.com" in emails
    assert "bob@example.com"   in emails
    assert "carol@example.com" in emails
    assert wl2.count() == 3
    checks += 1; print("✅ 4 persisted to disk; reload gives 3 entries")
except Exception as e:
    print("❌ 4:", e)

# 5 — CSV export has header and all emails
try:
    wl = Waitlist(wl_path)
    csv_out = wl.export_csv()
    lines = csv_out.strip().split("\n")
    assert "email" in lines[0], f"expected header, got: {lines[0]}"
    assert "alice@example.com" in csv_out
    assert "bob@example.com"   in csv_out
    assert len(lines) == 4      # header + 3 rows
    checks += 1; print("✅ 5 CSV has header + 3 data rows with correct emails")
except Exception as e:
    print("❌ 5:", e)

import os
try: os.unlink(wl_path)
except: pass

print(f"\n{checks}/5 checks passed!")
